In [0]:
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
import mlflow
import mlflow.sklearn
import pandas as pd


df_gold = spark.read.table("gold_credit_features")

pdf = df_gold.toPandas()


feature_cols = [c for c in pdf.columns if c != 'default']
X = pdf[feature_cols]
y = pdf['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [0]:
from mlflow.models.signature import infer_signature

with mlflow.start_run(run_name="gradient_boosting_v1"):
    
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    signature = infer_signature(X_train, y_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("auc", auc)
    
    mlflow.sklearn.log_model(
        pipeline,
        "credit_risk_model",
        registered_model_name="credit_risk_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )
    
    print(f"AUC：{auc:.4f}")
    print(classification_report(y_test, y_pred))